# 🔥 ФЕНИКС v32.0 — Sobol Monte Carlo Engine
## Worst-of Phoenix Autocallable Pricing

**Features:**
- Sobol QMC (quasi-random) — 500K симуляций
- Memory coupon pricing
- Vectorized GBM с Cholesky decomposition
- Google Drive backup
- ClickHouse кэширование
- xfinlink + yfinance data sources

⚡ Рекомендуется: GPU Runtime (T4) для максимальной скорости

In [ ]:
# 📦 Установка зависимостей
!pip install -q numpy scipy pandas yfinance clickhouse-connect plotly supabase

In [ ]:
# 📁 Google Drive монтирование
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_PATH = '/content/drive/MyDrive/phoenix_data/'
    import os
    os.makedirs(GDRIVE_PATH, exist_ok=True)
    print('✅ Google Drive смонтирован:', GDRIVE_PATH)
except Exception as e:
    GDRIVE_PATH = './phoenix_data/'
    import os
    os.makedirs(GDRIVE_PATH, exist_ok=True)
    print('⚠️ Google Drive не доступен, локальная папка:', GDRIVE_PATH)

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import qmc, norm
from datetime import datetime, timedelta
import time
import hashlib
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

# ClickHouse (опционально)
try:
    import clickhouse_connect
    CH_CLIENT = clickhouse_connect.get_client(
        host='mz5xp6056a.us-east1.gcp.clickhouse.cloud',
        port=8443,
        user='default',
        password='nSnvOjKP~2s53',
        secure=True
    )
    CH_AVAILABLE = True
    print('✅ ClickHouse Cloud подключен')
except Exception as e:
    CH_AVAILABLE = False
    print(f'⚠️ ClickHouse не доступен: {e}')

print('='*70)
print('🔥 ФЕНИКС v32.0 — SOBOL MONTE CARLO ENGINE')
print('='*70)

In [ ]:
# ============================================================
# КОНФИГУРАЦИЯ
# ============================================================
CONFIG = {
    'coupon': 0.065,                # 6.5% квартальный (26% годовых)
    'barrier': 0.65,                # 65% барьер
    'horizon_days': 504,            # 2 года
    'obs_days': [63, 126, 189, 252, 315, 378, 441, 504],
    'n_sims': 500_000,              # 500k симуляций (Colab GPU can handle this)
    'lookback_years': 2,
}

# 🎯 Твоя корзина (можно менять)
BASKET = ["AAPL", "MSFT", "GOOGL", "AMZN"]

basket_str = ', '.join(BASKET)
print(f'📊 КОРЗИНА: {basket_str}')
print(f'🎯 СТАВКА: {CONFIG["coupon"]*4*100:.0f}% годовых')
print(f'🛡️ БАРЬЕР: {CONFIG["barrier"]*100:.0f}%')
print(f'🎲 СИМУЛЯЦИЙ: {CONFIG["n_sims"]:,}')

In [ ]:
# ============================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================
def load_prices(tickers, days_back=730):
    """Загрузка цен через yfinance."""
    end = datetime.now()
    start = end - timedelta(days=days_back)
    data = {}
    for t in tickers:
        try:
            d = yf.download(t, start=start, end=end, progress=False)['Close']
            if len(d) > 200:
                data[t] = d.values.flatten()
                print(f'  ✅ {t}: {len(d)} дней')
            else:
                print(f'  ⚠️ {t}: только {len(d)} дней (нужно > 200)')
        except Exception as e:
            print(f'  ❌ {t}: {e}')
    return data

print('📥 Загрузка цен...')
prices_data = load_prices(BASKET)
print(f'\n✅ Загружено {len(prices_data)}/{len(BASKET)} тикеров')

In [ ]:
# ============================================================
# SOBOL MC СИМУЛЯЦИЯ
# ============================================================
def simulate_basket(basket, prices_data, config):
    """Векторизованная Sobol MC симуляция worst-of Phoenix."""
    min_len = min(len(prices_data[t]) for t in basket)
    prices_array = np.array([prices_data[t][:min_len] for t in basket]).T
    returns = np.diff(np.log(prices_array), axis=0)

    mu = returns.mean(axis=0)
    cov = np.cov(returns.T)
    L = np.linalg.cholesky(cov)
    n_assets = len(basket)
    n_sims = config['n_sims']
    horizon = config['horizon_days']

    # Sobol или pseudo-random
    dim = horizon * n_assets
    max_sobol_dim = 21201  # Colab has more memory than Streamlit Cloud

    print(f'  Размерность: {dim} (horizon={horizon} × assets={n_assets})')

    if dim <= max_sobol_dim:
        print(f'  🎲 Sobol QMC ({n_sims:,} путей, {dim} dims)...')
        n_pow2 = 1 << (n_sims - 1).bit_length()
        sampler = qmc.Sobol(d=dim, scramble=True)
        sobol_samples = sampler.random(n=n_pow2)[:n_sims]
        z_all = norm.ppf(np.clip(sobol_samples, 1e-8, 1 - 1e-8)).reshape(n_sims, horizon, n_assets)
        del sobol_samples
        method = 'Sobol QMC'
    else:
        print(f'  🎲 Pseudo-random ({n_sims:,} путей, {dim} dims > {max_sobol_dim})...')
        z_all = np.random.standard_normal((n_sims, horizon, n_assets))
        method = 'Pseudo-random'

    dt = 1 / 252
    sqrt_dt = np.sqrt(dt)

    # Vectorized GBM
    inc = np.einsum('ij,skj->ski', L, z_all) * sqrt_dt + mu * dt
    prices = np.exp(np.cumsum(inc, axis=1))
    ones = np.ones((n_sims, 1, n_assets))
    prices = np.concatenate([ones, prices], axis=1)
    worst = np.min(prices, axis=2)

    # Memory coupon
    total_coupons = np.zeros(n_sims)
    memory = np.zeros(n_sims, dtype=int)

    for idx in config['obs_days'][:-1]:
        paid = worst[:, idx] >= 1.0
        total_coupons[paid] += config['coupon'] * (1 + memory[paid])
        memory[paid] = 0
        memory[~paid] += 1

    final = worst[:, -1]
    capital_loss = final < config['barrier']
    principal = np.ones(n_sims)
    principal[capital_loss] = final[capital_loss]

    last_paid = worst[:, config['obs_days'][-1]] >= 1.0
    total_coupons[last_paid] += config['coupon'] * memory[last_paid]

    payoffs = principal + total_coupons

    # Per-ticker finals
    per_ticker = {}
    for i, t in enumerate(basket):
        finals = prices[:, -1, i]
        per_ticker[t] = {
            'mean': float(np.mean(finals) * 100),
            'std': float(np.std(finals) * 100),
            'min': float(np.min(finals) * 100),
            'max': float(np.max(finals) * 100),
            'var_95': float(np.percentile(finals, 5) * 100),
            'p_barrier': float(np.mean(finals < config['barrier']) * 100),
        }

    return {
        'avg_payoff': float(np.mean(payoffs)),
        'p_loss': float(np.mean(payoffs < 1)),
        'payoffs': payoffs,
        'method': method,
        'per_ticker': per_ticker,
        'timestamp': datetime.now().isoformat(),
    }

print('🚀 Запуск симуляции...')
t0 = time.time()
result = simulate_basket(BASKET, prices_data, CONFIG)
elapsed = time.time() - t0
result['elapsed'] = elapsed
print(f'\n⚡ Симуляция завершена за {elapsed:.1f} сек ({CONFIG["n_sims"]/elapsed:,.0f} путей/сек)')

In [ ]:
# ============================================================
# РАСЧЁТ МЕТРИК
# ============================================================
payoffs = result['payoffs']
avg_payoff = result['avg_payoff']
p_loss = result['p_loss']

var_95 = np.percentile(payoffs, 5)
cvar_95 = np.mean(payoffs[payoffs <= var_95]) if np.sum(payoffs <= var_95) > 0 else var_95
var_99 = np.percentile(payoffs, 1)
cvar_99 = np.mean(payoffs[payoffs <= var_99]) if np.sum(payoffs <= var_99) > 0 else var_99

coupons_received = np.clip((payoffs - 1) / CONFIG['coupon'], 0, 8)

# Bootstrap CI
boot = [np.mean(payoffs[np.random.choice(len(payoffs), len(payoffs))] < 1) for _ in range(200)]
p_loss_std = np.std(boot)

metrics = {
    'avg_payoff': avg_payoff,
    'p_loss': p_loss,
    'p_loss_ci': (p_loss - 1.96*p_loss_std, p_loss + 1.96*p_loss_std),
    'var_95': var_95,
    'cvar_95': cvar_95,
    'var_99': var_99,
    'cvar_99': cvar_99,
    'p_all_coupons': float(np.mean(coupons_received >= 7.5)),
    'p_zero_coupons': float(np.mean(coupons_received < 1)),
    'mean_coupons': float(np.mean(coupons_received)),
}

print('='*70)
print('📊 РЕЗУЛЬТАТЫ ФЕНИКС v32.0')
print('='*70)
print(f'\n🏆 КОРЗИНА: {", ".join(BASKET)}')
print(f'🎲 Метод: {result["method"]}')
print(f'\n📈 ДОХОДНОСТЬ:')
print(f'   Средний payoff: {avg_payoff:.4f} (+{(avg_payoff-1)*100:.1f}%)')
print(f'   Годовая: {(avg_payoff**(1/2)-1)*100:.1f}%')
print(f'\n📉 РИСК:')
print(f'   P(loss): {p_loss:.1%} [CI: {metrics["p_loss_ci"][0]:.1%} – {metrics["p_loss_ci"][1]:.1%}]')
print(f'   VaR 95%: {var_95:.4f} (потеря {(1-var_95)*100:.1f}%)')
print(f'   CVaR 95%: {cvar_95:.4f}')
print(f'   VaR 99%: {var_99:.4f}')
print(f'   CVaR 99%: {cvar_99:.4f}')
print(f'\n🎯 КУПОНЫ:')
print(f'   Все 8: {metrics["p_all_coupons"]:.1%}')
print(f'   Среднее: {metrics["mean_coupons"]:.1f} из 8')
print(f'   0 купонов: {metrics["p_zero_coupons"]:.1%}')
print(f'\n⚡ {CONFIG["n_sims"]:,} симуляций за {elapsed:.1f} сек')

In [ ]:
# ============================================================
# PER-TICKER АНАЛИЗ
# ============================================================
print('\n📋 PER-TICKER FINALS:')
print(f'{"Ticker":<8} {"Mean":>8} {"VaR95":>8} {"P(barr)":>8} {"Min":>8} {"Max":>8}')
print('-'*50)
for t in BASKET:
    d = result['per_ticker'][t]
    print(f'{t:<8} {d["mean"]:>7.1f}% {d["var_95"]:>7.1f}% {d["p_barrier"]:>7.1f}% {d["min"]:>7.1f}% {d["max"]:>7.1f}%')

In [ ]:
# ============================================================
# ВИЗУАЛИЗАЦИЯ
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(rows=2, cols=2,
    subplot_titles=('Payoff Distribution', 'Coupons Distribution',
                    'Per-Ticker Final Returns', 'Cumulative Payoff CDF'))

# Payoff histogram
fig.add_trace(go.Histogram(x=payoffs, nbinsx=100, name='Payoff',
    marker_color='#ffb000', opacity=0.8), row=1, col=1)
fig.add_vline(x=1.0, line_dash='dash', line_color='red', row=1, col=1)
fig.add_vline(x=var_95, line_dash='dash', line_color='cyan', row=1, col=1)

# Coupons histogram
fig.add_trace(go.Histogram(x=coupons_received, nbinsx=9, name='Coupons',
    marker_color='#34c759', opacity=0.8), row=1, col=2)

# Per-ticker bar
tickers_names = list(result['per_ticker'].keys())
means = [result['per_ticker'][t]['mean'] for t in tickers_names]
vars95 = [result['per_ticker'][t]['var_95'] for t in tickers_names]
fig.add_trace(go.Bar(x=tickers_names, y=means, name='Mean %',
    marker_color='#6db6ff'), row=2, col=1)
fig.add_trace(go.Bar(x=tickers_names, y=vars95, name='VaR 95%',
    marker_color='#ff3b30'), row=2, col=1)
fig.add_hline(y=65, line_dash='dash', line_color='yellow', row=2, col=1)

# CDF
sorted_payoffs = np.sort(payoffs)
cdf = np.arange(1, len(sorted_payoffs)+1) / len(sorted_payoffs)
fig.add_trace(go.Scatter(x=sorted_payoffs[::100], y=cdf[::100],
    name='CDF', line=dict(color='#fa8000')), row=2, col=2)
fig.add_vline(x=1.0, line_dash='dash', line_color='red', row=2, col=2)

fig.update_layout(
    height=700,
    template='plotly_dark',
    title_text=f'🔥 ФЕНИКС v32.0 — {", ".join(BASKET)} — {CONFIG["n_sims"]:,} MC paths',
    showlegend=False,
    paper_bgcolor='#000',
    plot_bgcolor='#0a0a0a',
)
fig.show()

In [ ]:
# ============================================================
# СОХРАНЕНИЕ В GOOGLE DRIVE
# ============================================================
report = {
    'timestamp': datetime.now().isoformat(),
    'basket': BASKET,
    'config': CONFIG,
    'metrics': metrics,
    'per_ticker': result['per_ticker'],
    'method': result['method'],
    'elapsed_sec': elapsed,
    'n_sims': CONFIG['n_sims'],
}

filename = f'phoenix_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
filepath = os.path.join(GDRIVE_PATH, filename)

with open(filepath, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f'💾 Отчёт сохранён: {filepath}')
print(f'📁 Папка: {GDRIVE_PATH}')

# Список сохранённых отчётов
saved = [f for f in os.listdir(GDRIVE_PATH) if f.startswith('phoenix_report')]
print(f'\n📋 Всего отчётов: {len(saved)}')
for f in sorted(saved)[-5:]:
    print(f'  📄 {f}')

In [ ]:
# ============================================================
# CLICKHOUSE — QUANTUM RISK DATA
# ============================================================
if CH_AVAILABLE:
    print('📊 ClickHouse Cloud — MIT Quantum Returns')
    print('='*50)

    # Total count
    total = CH_CLIENT.query('SELECT count() FROM mit_quantum_returns').result_rows[0][0]
    print(f'Total simulations: {total:,}')

    # Worst-of VaR
    q = '''
    SELECT
        quantile(0.05)(wo_final) AS var_95,
        quantile(0.01)(wo_final) AS var_99,
        avg(wo_final) AS mean_wo,
        countIf(wo_final < 0.65) / count() AS p_barrier
    FROM (
        SELECT simulation_id, min(final_return) AS wo_final
        FROM mit_quantum_returns
        GROUP BY simulation_id
    )
    '''
    row = CH_CLIENT.query(q).result_rows[0]
    print(f'\nWorst-of Portfolio:')
    print(f'  VaR 95%: {row[0]*100:.1f}%')
    print(f'  VaR 99%: {row[1]*100:.1f}%')
    print(f'  Mean: {row[2]*100:.1f}%')
    print(f'  P(barrier 65%): {row[3]*100:.1f}%')

    # Per-ticker
    q2 = '''
    SELECT
        ticker,
        quantile(0.05)(final_return) AS var_95,
        avg(final_return) AS mean_ret,
        countIf(final_return < 0.65) / count() AS p_barrier,
        count() AS n
    FROM mit_quantum_returns
    GROUP BY ticker
    ORDER BY var_95
    '''
    rows = CH_CLIENT.query(q2).result_rows
    print(f'\nPer-Ticker:')
    print(f'{"Ticker":<8} {"VaR95":>8} {"Mean":>8} {"P(barr)":>8} {"N":>8}')
    for r in rows:
        print(f'{r[0]:<8} {r[1]*100:>7.1f}% {r[2]*100:>7.1f}% {r[3]*100:>7.1f}% {r[4]:>8,}')
else:
    print('⚠️ ClickHouse не подключен — пропуск')

---
## 🎉 ФЕНИКС v32.0 — ГОТОВО

Результаты сохранены в Google Drive. Дашборд автоматически подтянет данные при следующем обновлении.

**Dashboard:** https://meq9hf6guncl5mbpflmwzp.streamlit.app/

**Параметры для изменения:**
- `BASKET` — список тикеров
- `CONFIG['n_sims']` — количество симуляций (до 1M на GPU)
- `CONFIG['coupon']` — купон (0.065 = 6.5%/кв)
- `CONFIG['barrier']` — барьер (0.65 = 65%)